# Волна 2: категория на складе и веса участников ансамбля

База - "v5 + LightGBM" (лучший на сегодня: `05_lgbm`). Развиваем то, что сработало.

- Категория на складе (`features.GROUPS["категория на складе"]`). Тренд категории был первым нашим выигрышем, но рост категории разный по складам: 1_4 выросла от ×1.4 до ×3.1 в зависимости от склада (EDA, раздел 8). Новые признаки: 4 недели к 24 у пары "склад × категория" 1-го и 2-го уровня; доля пар категории с продажами за 4 недели - всего и на складе; она же к доле за 24 недели (категория разогревается или остывает). У обеих моделей, всех трёх участников.
- Веса участников (поверх тех же моделей, бесплатно): сейчас простое среднее, у LightGBM треть веса. Варианты: LightGBM - половина веса, пятая часть; LightGBM только в классификаторе или только в регрессоре.

Вариант проходит, если он не хуже базы (посчитанной в этом же запуске) на обоих сезонных фолдах, а на `recent` хуже не больше чем на 0.005; из прошедших лучший - по среднему двух сезонных. Таблица всех вариантов сохраняется в `grid_<имя>.csv` - потом лучшие идеи собираются вместе.

In [1]:
import sys; sys.path[:0] = ["../src", "../../src"]   # код проекта - в src/ (ноутбук в notebooks/ или notebooks/05_experiments/)

import pandas as pd
import core
from core import *
import classifier, regressor, pipeline

NAME = "05_whcat"
# быстрее, чем в 03/04: на Kaggle - каждая 2-я неделя отсчёта (так валидировался v5); деревьев - по 400, как в модулях
if ON_KAGGLE:
    classifier.ORIGIN_STEP = regressor.ORIGIN_STEP = 2
V5 = {"threshold": 0.5, "integer": "smape", "ensemble": [{}, {"random_seed": 7}]}
LGB = {"library": "lightgbm"}
V5L = {**V5, "ensemble": [{}, {"random_seed": 7}, LGB]}                                 # v5 + LightGBM (05_lgbm)
SEASON = ["season2025", "season2024"]
sales, stock = load_data()
classifier.PARAMS, regressor.PARAMS, classifier.HALF_LIFE, classifier.ORIGIN_STEP

({'iterations': 400,
  'learning_rate': 0.1,
  'depth': 6,
  'loss_function': 'Logloss',
  'max_ctr_complexity': 1,
  'random_seed': 42,
  'verbose': 0,
  'allow_writing_files': False,
  'thread_count': -1,
  'one_hot_max_size': 64,
  'task_type': 'GPU'},
 {'iterations': 400,
  'learning_rate': 0.1,
  'depth': 6,
  'loss_function': 'RMSE',
  'max_ctr_complexity': 1,
  'random_seed': 42,
  'verbose': 0,
  'allow_writing_files': False,
  'thread_count': -1,
  'task_type': 'GPU'},
 26,
 2)

## Варианты

Параметры метода `pipeline`; база для сравнения - "v5 + LightGBM".

In [2]:
BASE_NAME = "v5 + LightGBM"
from features import GROUPS
NEW = GROUPS["категория на складе"][0]
CF, RF = {"features": classifier.FEATURES + NEW}, {"features": regressor.FEATURES + NEW}
VARIANTS = {
    "v5 + LightGBM": V5L,
    "+ категория на складе": {**V5, "ensemble": [{"clf": CF, "reg": RF}, {"random_seed": 7, "clf": CF, "reg": RF},
                                                  {**LGB, "clf": CF, "reg": RF}]},
    "вес LightGBM 1/2": {**V5, "ensemble": [{}, {"random_seed": 7}, {**LGB, "weight": 2}]},
    "вес LightGBM 1/5": {**V5, "ensemble": [{}, {"random_seed": 7}, {**LGB, "weight": 0.5}]},
    "LightGBM только в классификаторе": {**V5, "ensemble": [{}, {"random_seed": 7}, {**LGB, "reg_weight": 0}]},
    "LightGBM только в регрессоре": {**V5, "ensemble": [{}, {"random_seed": 7}, {**LGB, "clf_weight": 0}]},
}
list(VARIANTS)

['v5 + LightGBM',
 '+ категория на складе',
 'вес LightGBM 1/2',
 'вес LightGBM 1/5',
 'LightGBM только в классификаторе',
 'LightGBM только в регрессоре']

## Сетка на трёх фолдах

Модели учатся один раз на фолд и набор настроек, варианты перебираются поверх кэша. Если вариант не посчитается, он пропускается - остальное считается.

In [3]:
rows = []
for fold, cut in FOLDS.items():
    pairs, s_cut, st_cut, truth, base = core._fold_data(fold, cut)
    for name, params in VARIANTS.items():
        try:
            pred = make_forecast(s_cut, st_cut, "pipeline", pairs=pairs, **params)
        except Exception as e:
            print(f"{fold}, {name}: пропущен - {type(e).__name__}: {e}", flush=True)
            continue
        m = compute_metrics(truth, pred)
        rows.append({"вариант": name, "фолд": fold, **m, "score": score_minmax(m, base),
                     "прогноз / факт": pred.quantity.sum() / truth.quantity.sum()})
    print(fold, "готов", flush=True)
grid = pd.DataFrame(rows)
grid.to_csv(OUT_DIR / f"grid_{NAME}.csv", index=False)
grid.pivot_table(index="вариант", columns="фолд", values=["smape", "wape", "bias", "rmse"]).round(2)

[LightGBM] [Warning] Met negative value in categorical features, will convert it to NaN
[LightGBM] [Warning] Met negative value in categorical features, will convert it to NaN
[LightGBM] [Warning] Met negative value in categorical features, will convert it to NaN


[LightGBM] [Warning] Met negative value in categorical features, will convert it to NaN
[LightGBM] [Warning] Met negative value in categorical features, will convert it to NaN
[LightGBM] [Warning] Met negative value in categorical features, will convert it to NaN


season2025 готов


[LightGBM] [Warning] Met negative value in categorical features, will convert it to NaN
[LightGBM] [Warning] Met negative value in categorical features, will convert it to NaN
[LightGBM] [Warning] Met negative value in categorical features, will convert it to NaN


[LightGBM] [Warning] Met negative value in categorical features, will convert it to NaN
[LightGBM] [Warning] Met negative value in categorical features, will convert it to NaN
[LightGBM] [Warning] Met negative value in categorical features, will convert it to NaN


season2024 готов


recent готов


bias                         rmse  \
фолд                             recent season2024 season2025 recent   
вариант                                                                
+ категория на складе             75.40      84.97      79.35   3.38   
LightGBM только в классификаторе  76.21      86.45      81.04   3.35   
LightGBM только в регрессоре      75.99      86.60      80.58   3.34   
v5 + LightGBM                     75.67      86.45      80.59   3.34   
вес LightGBM 1/2                  75.45      86.55      80.37   3.35   
вес LightGBM 1/5                  76.07      86.48      80.69   3.34   

                                                        smape             \
фолд                             season2024 season2025 recent season2024   
вариант                                                                    
+ категория на складе                  1.92       2.76  58.58      42.75   
LightGBM только в классификаторе       1.93       2.64  58.67      42.81   
LightGBM только в регрессоре           1.88       2.64  58.64      42.85   
v5 + LightGBM                          1.88       2.64  58.62      42.80   
вес LightGBM 1/2                       1.87       2.65  58.59      42.82   
вес LightGBM 1/5                       1.90       2.64  58.67      42.79   

                                              wape                        
фолд                             season2025 recent season2024 season2025  
вариант                                                                   
+ категория на складе                 57.58  47.21      62.38      51.52  
LightGBM только в классификаторе      57.44  47.09      62.89      51.13  
LightGBM только в регрессоре          57.58  47.01      62.62      51.11  
v5 + LightGBM                         57.47  47.01      62.58      51.06  
вес LightGBM 1/2                      57.50  47.06      62.50      51.08  
вес LightGBM 1/5                      57.48  47.04      62.71      51.08

## Выбор

Вариант проходит, если он не хуже базы (посчитанной в этом же запуске) на обоих сезонных фолдах, а на `recent` хуже не больше чем на 0.005; из прошедших лучший - по среднему двух сезонных. Таблица всех вариантов сохраняется в `grid_<имя>.csv` - потом лучшие идеи собираются вместе.

In [4]:
scores = grid.pivot_table(index="вариант", columns="фолд", values="score")[list(FOLDS)]
BASE = scores.loc[BASE_NAME]
scores["по сезону"] = scores[SEASON].mean(axis=1)
scores["проходит"] = (scores[SEASON] <= BASE[SEASON]).all(axis=1) & (scores["recent"] <= BASE["recent"] + 0.005)
scores.loc[BASE_NAME, "проходит"] = False
passed = scores[scores["проходит"]]
BEST = passed["по сезону"].idxmin() if len(passed) else None
print(BASE_NAME, "в этом запуске:", BASE[list(FOLDS)].round(4).to_dict(), "| по сезону", round(BASE[SEASON].mean(), 4))
print("лучший вариант:", BEST, scores.loc[BEST, list(FOLDS) + ["по сезону"]].astype(float).round(4).to_dict() if BEST else "- не прошёл ни один")
scores.sort_values("по сезону").round(4)

v5 + LightGBM в этом запуске: {'season2025': 0.8075, 'season2024': 0.8196, 'recent': 0.7852} | по сезону 0.8135
лучший вариант: None - не прошёл ни один


фолд,season2025,season2024,recent,по сезону,проходит
вариант,,,,,
вес LightGBM 1/2,0.8079,0.8184,0.7851,0.8132,False
v5 + LightGBM,0.8075,0.8196,0.7852,0.8135,False
LightGBM только в регрессоре,0.8084,0.8203,0.7856,0.8144,False
вес LightGBM 1/5,0.8077,0.8211,0.7860,0.8144,False
LightGBM только в классификаторе,0.8081,0.8235,0.7866,0.8158,False
+ категория на складе,0.8145,0.8196,0.7866,0.8171,False
